**Cell 1: Import Libraries and Load Email Data**  
This cell imports the required libraries and loads the merged dataset, then filters it to only include email-related columns plus the churn outcome.  
- **Purpose**: Prepare the notebook environment and load the email dataset for hypothesis testing.  
- **Key libraries**: `pandas`, `numpy`, `scipy.stats`, `statsmodels.stats.multitest`.  
- **No hypothesis test here** — this is data preparation.  
- **Expected output**: a table preview of the selected email features.

In [5]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind, mannwhitneyu, chi2_contingency
from statsmodels.stats.multitest import multipletests

df = pd.read_csv("../../data/04_merged/final_merged_features.csv")

# Select only email related columns
email_cols = [
    "co_ref", "has_emails_data", "total_interactions", "interactions_14_days", 
    "pre_renewal_interactions", "last_moment_engagement_ratio", "complaints", 
    "negative_experience", "support_issues", "financial_stress", "price_dissatisfaction", 
    "avg_sentiment", "agent_followups", "engagement_score", "prospect_outcome"
]

df = df[email_cols].dropna(subset=["prospect_outcome"])

df.head()

,co_ref,has_emails_data,total_interactions,interactions_14_days,pre_renewal_interactions,last_moment_engagement_ratio,complaints,negative_experience,support_issues,financial_stress,price_dissatisfaction,avg_sentiment,agent_followups,engagement_score,prospect_outcome
0,AA0794,1,9.0,3.0,3.0,0.3,0.0,0.0,0.0,0.0,0.0,50.0,9.0,0.0,Won
1,AA0794,1,9.0,3.0,3.0,0.3,0.0,0.0,0.0,0.0,0.0,50.0,9.0,0.0,Won
2,AA0794,1,9.0,3.0,3.0,0.3,0.0,0.0,0.0,0.0,0.0,50.0,9.0,0.0,Won
3,AA0794,1,9.0,3.0,3.0,0.3,0.0,0.0,0.0,0.0,0.0,50.0,9.0,0.0,Won
4,AA0794,1,9.0,3.0,3.0,0.3,0.0,0.0,0.0,0.0,0.0,50.0,9.0,0.0,Won


**Cell 2: Create Churn Target**  
This cell converts the prospect outcome into a binary churn variable, where 1 indicates "Churned" and 0 indicates not churned.  
- **Purpose**: Create the dependent variable used for hypothesis testing.  
- **No hypothesis testing here** — this is target creation.  
- **Expected output**: counts for churned vs. non-churned cases.

In [6]:
df["target"] = (df["prospect_outcome"] == "Churned").astype(int)

df["target"].value_counts()

target
0    102310
1     15011
Name: count, dtype: int64

**Cell 3: Define Email Feature Columns**  
This cell defines the numerical email features and categorical features to be tested for churn association.  
- **Purpose**: Select the feature set for hypothesis testing.  
- **No hypothesis testing here** — this is feature selection.  
- **Expected output**: no direct output, just the lists of features.

In [ ]:
num_cols = [
    "emails_sent_14d",
    "emails_opened_14d",
    "email_open_rate_14d",
    "email_click_rate_14d",
    "response_rate_14d",
    "bounce_rate_14d",
    "last_email_gap"
]

cat_cols = [
    "unsubscribe_flag",
    "campaign_type_mode"
]

**Cell 4: Hypothesis Testing for Numerical Email Features**  
This cell runs statistical tests on each numerical email feature to compare churn and non-churn groups.  
- **Null Hypothesis (H₀)**: There is no difference in the feature distribution between churn and non-churn customers.  
- **Alternative Hypothesis (H₁)**: There is a difference in the feature distribution between churn and non-churn customers.  
- **Tests used**:  
  - Welch t-test for mean differences  
  - Mann-Whitney U test for distribution differences  
  - Cohen's d for effect size  
- **Expected output**: A DataFrame with means, p-values, and effect sizes for each numerical feature.

In [8]:
num_results = []

for col in num_cols:
    churn = df[df["target"] == 1][col].dropna()
    non_churn = df[df["target"] == 0][col].dropna()

    if len(churn) > 1 and len(non_churn) > 1:
        # Welch T-test
        t_stat, t_p = ttest_ind(churn, non_churn, equal_var=False)

        # Mann Whitney
        u_stat, u_p = mannwhitneyu(churn, non_churn, alternative="two-sided")

        # Cohen's d
        pooled_std = np.sqrt((churn.var() + non_churn.var()) / 2)
        effect_size = (
            (churn.mean() - non_churn.mean()) / pooled_std
            if pooled_std != 0 else 0
        )

        num_results.append({
            "feature": col,
            "churn_mean": churn.mean(),
            "non_churn_mean": non_churn.mean(),
            "ttest_p_value": t_p,
            "mannwhitney_p_value": u_p,
            "effect_size": effect_size
        })

num_results = pd.DataFrame(num_results)
num_results

,feature,churn_mean,non_churn_mean,ttest_p_value,mannwhitney_p_value,effect_size
0,total_interactions,9.862711,11.723227,2.021787e-187,1.306052e-240,-0.309562
1,interactions_14_days,2.573941,3.150201,4.231314e-137,9.988360e-190,-0.254614
2,pre_renewal_interactions,2.346154,2.023293,1.644943e-53,5.484586e-26,0.157937
3,last_moment_engagement_ratio,0.230357,0.237127,8.512276e-05,3.276094e-19,-0.042405
4,complaints,0.987145,0.626844,5.331921e-71,9.544865e-133,0.205271
5,negative_experience,2.464829,1.520539,5.040637e-203,2.326647e-296,0.352613
6,support_issues,1.189737,0.952774,2.040653e-25,4.085333e-38,0.115624
7,financial_stress,1.196216,0.463174,4.223488e-271,0.000000e+00,0.436568
8,price_dissatisfaction,0.940765,0.879980,3.191894e-03,6.199175e-11,0.031157
9,avg_sentiment,41.009950,50.540609,0.000000e+00,0.000000e+00,-0.484904


**Cell 5: Hypothesis Testing for Categorical Email Features**  
This cell checks whether any categorical email features are associated with churn using a chi-square test.  
- **Null Hypothesis (H₀)**: The categorical feature is independent of churn.  
- **Alternative Hypothesis (H₁)**: The categorical feature is associated with churn.  
- **Test used**: Chi-square test of independence.  
- **Expected output**: A DataFrame with p-values for each tested categorical feature.

In [9]:
cat_results = []

for col in cat_cols:
    cont_table = pd.crosstab(df[col], df["target"])

    if cont_table.shape[0] > 1 and cont_table.shape[1] > 1:
        chi2, p, dof, expected = chi2_contingency(cont_table)

        cat_results.append({
            "feature": col,
            "chi2_p_value": p
        })

cat_results = pd.DataFrame(cat_results)
cat_results

""


**Cell 6: Adjust P-Values for Multiple Testing**  
This cell corrects the numerical test p-values using the Benjamini-Hochberg false discovery rate method.  
- **Purpose**: Control the false positive rate across multiple feature tests.  
- **Key output**: `adjusted_p` and `significant` columns.  
- **Decision rule**: A feature is considered significant when `adjusted_p < 0.05`.

In [10]:
num_results["adjusted_p"] = multipletests(
    num_results["mannwhitney_p_value"],
    method="fdr_bh"
)[1]

num_results["significant"] = num_results["adjusted_p"] < 0.05

num_results.sort_values("adjusted_p")

,feature,churn_mean,non_churn_mean,ttest_p_value,mannwhitney_p_value,effect_size,adjusted_p,significant
7,financial_stress,1.196216,0.463174,4.223488e-271,0.000000e+00,0.436568,0.000000e+00,True
9,avg_sentiment,41.009950,50.540609,0.000000e+00,0.000000e+00,-0.484904,0.000000e+00,True
5,negative_experience,2.464829,1.520539,5.040637e-203,2.326647e-296,0.352613,9.306588e-296,True
0,total_interactions,9.862711,11.723227,2.021787e-187,1.306052e-240,-0.309562,3.918155e-240,True
1,interactions_14_days,2.573941,3.150201,4.231314e-137,9.988360e-190,-0.254614,2.397206e-189,True
4,complaints,0.987145,0.626844,5.331921e-71,9.544865e-133,0.205271,1.908973e-132,True
6,support_issues,1.189737,0.952774,2.040653e-25,4.085333e-38,0.115624,7.003427e-38,True
2,pre_renewal_interactions,2.346154,2.023293,1.644943e-53,5.484586e-26,0.157937,8.226879e-26,True
3,last_moment_engagement_ratio,0.230357,0.237127,8.512276e-05,3.276094e-19,-0.042405,4.368126e-19,True
10,agent_followups,6.445084,6.938704,1.228817e-11,8.976374e-13,-0.071322,1.077165e-12,True


**Cell 7: Display Significant Email Features**  
This cell filters the numerical results to show only email features that remain significant after p-value correction.  
- **Purpose**: Highlight the strongest churn-related email metrics.  
- **Expected output**: a table of significant features with corrected p-values.

In [11]:
significant_features = num_results[num_results["significant"] == True]
significant_features

,feature,churn_mean,non_churn_mean,ttest_p_value,mannwhitney_p_value,effect_size,adjusted_p,significant
0,total_interactions,9.862711,11.723227,2.021787e-187,1.306052e-240,-0.309562,3.918155e-240,True
1,interactions_14_days,2.573941,3.150201,4.231314e-137,9.988360e-190,-0.254614,2.397206e-189,True
2,pre_renewal_interactions,2.346154,2.023293,1.644943e-53,5.484586e-26,0.157937,8.226879e-26,True
3,last_moment_engagement_ratio,0.230357,0.237127,8.512276e-05,3.276094e-19,-0.042405,4.368126e-19,True
4,complaints,0.987145,0.626844,5.331921e-71,9.544865e-133,0.205271,1.908973e-132,True
5,negative_experience,2.464829,1.520539,5.040637e-203,2.326647e-296,0.352613,9.306588e-296,True
6,support_issues,1.189737,0.952774,2.040653e-25,4.085333e-38,0.115624,7.003427e-38,True
7,financial_stress,1.196216,0.463174,4.223488e-271,0.000000e+00,0.436568,0.000000e+00,True
8,price_dissatisfaction,0.940765,0.879980,3.191894e-03,6.199175e-11,0.031157,6.762737e-11,True
9,avg_sentiment,41.009950,50.540609,0.000000e+00,0.000000e+00,-0.484904,0.000000e+00,True


**Cell 8: Save Hypothesis Test Results**  
This cell exports the numerical and categorical results to CSV files for reporting or further analysis.  
- **Purpose**: Persist the final results to the `reports` directory.  
- **No hypothesis testing here** — this is an output/save step.  
- **Expected output**: two saved report files with the test results.

In [12]:
num_results.to_csv(
    "../../reports/email_hypothesis_results.csv",
    index=False
)

cat_results.to_csv(
    "../../reports/email_categorical_hypothesis_results.csv",
    index=False
)